In [1]:
import os
from pathlib import Path, PureWindowsPath

import numpy as np
import pandas as pd
import pyopencl as cl
platform = cl.get_platforms()[0]
device = platform.get_devices()[0]
ctx = cl.Context(devices=[device])
import apoc
import pyclesperanto as cle
from skimage.io import imread, imshow, imsave

import time
import re


In [ ]:
#cle.select_device("RTX")

In [5]:
def reduce_feature_df(feature_df: pd.DataFrame, n_drop: int = 1) -> pd.DataFrame:
    """Description"""
    trimmed_df = feature_df[:-n_drop]
    
    return trimmed_df

In [6]:
def build_feature_str(feature_df: pd.DataFrame) -> str:
    """
    Takes a DataFrame with a 'feature' column and returns a space-separated feature string.
    
    Parameters
    ----------
    feature_df : pd.DataFrame
        DataFrame containing a 'feature' column with entries like 'gaussian_blur=1'.
        
    Returns
    -------
    str
        Feature specification string suitable for APOC PixelClassifier.
    """
    return ' '.join(feature_df['feature'].tolist())

In [7]:
def train_apoc_model(features: str, labels: list, images: list, cl_filename: str, num_ensembles: int = 250, max_depth: int = 5) -> apoc.PixelClassifier:
    apoc.erase_classifier(cl_filename)
    clf = apoc.PixelClassifier(opencl_filename=cl_filename, num_ensembles=num_ensembles, max_depth=max_depth)
    clf.train(features=features, ground_truth=labels[0], image=images[0])
    for label, image in zip(labels[1:], images[1:]):
        clf.train(features=features, ground_truth=label, image=image, continue_training=True)
    return clf

In [8]:
def train_apoc_object(features: str, labels: list, images: list, cl_filename: str, num_ensembles: int = 250, max_depth: int = 5) -> apoc.ObjectSegmenter:
    apoc.erase_classifier(cl_filename)
    clf = apoc.ObjectSegmenter(opencl_filename=cl_filename, num_ensembles=num_ensembles, max_depth=max_depth)
    clf.train(features=features, ground_truth=labels[0], image=images[0])
    for label, image in zip(labels[1:], images[1:]):
        clf.train(features=features, ground_truth=label, image=image, continue_training=True)
    return clf

In [9]:
def build_feature_str(feature_df: pd.DataFrame) -> str:
    """
    Takes a DataFrame with a 'feature' column and returns a space-separated feature string.
    
    Parameters
    ----------
    feature_df : pd.DataFrame
        DataFrame containing a 'feature' column with entries like 'gaussian_blur=1'.
        
    Returns
    -------
    str
        Feature specification string suitable for APOC PixelClassifier.
    """
    return ' '.join(feature_df['feature'].tolist())

In [10]:
def cl_file_features(cl_file_path: Path) -> pd.DataFrame:
    """Description"""

    # Open cl file and take the feature_specification and feature_importance lines
    with open(cl_file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
        feature_lines = [line.strip() for line in lines if "feature_specification" in line or "feature_importance" in line]

    # Split features and importances, remove extra spaces
    feature_str = feature_lines[0].split("=", 1)[1].strip()
    feature_imp = feature_lines[1].split("=", 1)[1].strip()

    # Associate each feature with its importance
    feature_dic = {feature:importance for feature, importance in zip(feature_str.split(' '), map(float, feature_imp.split(',')))}
    feature_df = pd.DataFrame(list(feature_dic.items()), columns=["feature", "importance"]).sort_values(by="importance", ascending=False).reset_index(drop=True)

    # Extract filter type and radius from 'feature'
    feature_df[['filter', 'radius']] = feature_df['feature'].str.split('=', expand=True)
    feature_df['radius'] = feature_df['radius'].astype(float)

    return feature_df

In [8]:
def compute_dice_iou_with_ignore(
    ground_truth: np.ndarray,
    prediction: np.ndarray,
    foreground_label: int = 2,
    ignore_label: int = 0,
):
    """
    Compute Dice and IoU between two binary masks, ignoring a specific label (e.g., undefined border).
    
    Parameters:
    - ground_truth: Ground truth label image
    - prediction: Predicted label image
    - foreground_label: Label representing the foreground class
    - ignore_label: Label representing pixels to ignore during evaluation

    Returns:
    - dice: Dice coefficient
    - iou: Intersection over Union
    """

    # Define valid pixels (i.e., not ignored)
    valid_mask = ground_truth != ignore_label

    # Extract foreground masks
    gt = (ground_truth == foreground_label) & valid_mask
    pred = (prediction == foreground_label) & valid_mask

    intersection = np.logical_and(gt, pred).sum()
    union = np.logical_or(gt, pred).sum()

    if gt.sum() + pred.sum() == 0:
        dice = 1.0  # Perfect match if both are empty
    else:
        dice = 2.0 * intersection / (gt.sum() + pred.sum())

    iou = intersection / union if union != 0 else 1.0

    return dice, iou



In [9]:
def run_feature_elimination_loop(initial_features, labels, images, test_images, test_labels, output_dir, num_ensembles=250, max_depth=5, min_features=10):
    os.makedirs(output_dir, exist_ok=True)
    feature_string = initial_features
    feature_n = len(feature_string.split(' '))
    results = []
    iteration = 0
    while feature_n > min_features:
        cl_file = os.path.join(output_dir, f"pixel_classifier_iteration_{iteration}.cl")

        start_time = time.time()
        clf = train_apoc_model(features=feature_string, labels=labels, images=images, cl_filename=cl_file, num_ensembles=num_ensembles, max_depth=max_depth)
        duration = time.time() - start_time

        predictions = [np.asarray(clf.predict(test_image)) for test_image in test_images]
        # dice, iou = compute_dice_iou_with_ignore(test_label, prediction)
        dice = [compute_dice_iou_with_ignore(test_label, prediction)[0] for test_label, prediction in zip(test_labels, predictions)]
        iou = [compute_dice_iou_with_ignore(test_label, prediction)[1] for test_label, prediction in zip(test_labels, predictions)]

        results.append({
            "iteration": iteration,
            "num_features": feature_n,
            "features": feature_string,
            "cl_file": cl_file,
            "training_time": duration,
            "dice": dice,
            "iou": iou
        })

        feature_df = cl_file_features(cl_file)
        feature_reduced = reduce_feature_df(feature_df=feature_df)
        feature_string = build_feature_str(feature_df=feature_reduced)
        feature_n = len(feature_reduced)
        iteration += 1

    return pd.DataFrame(results)

In [11]:
# Input parameters

training_images_path = r"H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\training_data\images"
training_labels_path = r"H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\training_data\labels"
test_images_path = r"H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\test_data\images"
test_labels_path = r"H:\PROJECTS-03\Pablo\Testing\test_zyla_segmentation\test_data\labels"
image_ext = 'tiff'

In [12]:
training_images_list = list(Path(training_images_path).glob('*.tiff'))
training_labels_list = list(Path(training_labels_path).glob('*.tiff'))
test_images_list = list(Path(test_images_path).glob('*.tiff'))
test_labels_list = list(Path(test_labels_path).glob('*.tiff'))

In [13]:
# Load training data into GPU
training_images = [cle.push(imread(training_image.as_posix())) for training_image in training_images_list]
training_labels = [cle.push(imread(training_label.as_posix()).astype('uint8')) for training_label in training_labels_list]

# # Load training data into CPU
# training_images = [imread(training_image.as_posix()) for training_image in training_images_list]
# training_labels = [imread(training_label.as_posix()).astype('uint8') for training_label in training_labels_list]

In [14]:
# Load test data for accuracy metrics
test_images = [imread(test_image.as_posix()) for test_image in test_images_list]
test_labels = [imread(test_label.as_posix()) for test_label in test_labels_list]

In [46]:
# Inital parameters

radii = [1, 2, 3, 5, 10, 15, 20, 30]
filters = [
    "gaussian_blur",
    "mean_box",
    "top_hat_box",
    "maximum_box",
    "variance_box",
    "difference_of_gaussian",
    "sobel_of_gaussian_blur",
    "laplace_box_of_gaussian_blur",
    "small_hessian_eigenvalue_of_gaussian_blur",
    "large_hessian_eigenvalue_of_gaussian_blur",
]

initial_features = " ".join(
    f"{f}={r}" for f in filters for r in radii
)
initial_features = initial_features+' median_box=3 median_box=5 sobel_of_median_box=3 sobel_of_median_box=5'
num_ensembles=250
max_depth=5

In [ ]:
initial_features_df

In [29]:
initial_features = build_feature_str(initial_features_df)

In [48]:
results=run_feature_elimination_loop(initial_features=initial_features, 
                             labels=training_labels, 
                             images=training_images, 
                             test_images=test_images, 
                             test_labels=test_labels,
                             output_dir='./zyla_training', 
                             num_ensembles=250, 
                             max_depth=5, 
                             min_features=1)
results.to_csv('./training_zyla_df')

In [15]:
features_df = cl_file_features(Path('./zyla_training/pixel_classifier_iteration_40.cl'))
features_str = build_feature_str(features_df)

In [20]:
train_apoc_model?

Signature:
train_apoc_model(
    features: str,
    labels: list,
    images: list,
    cl_filename: str,
    num_ensembles: int = 250,
    max_depth: int = 5,
) -> apoc._pixel_classifier.PixelClassifier
Docstring: <no docstring>
File:      c:\users\pperez\appdata\local\temp\1\ipykernel_37476\3755655736.py
Type:      function


In [16]:
train_apoc_object(features=features_str, labels=training_labels, images=training_images, cl_filename='./object_classifier_features40.cl')

Classifier type: ObjectSegmenter
--- Random forest info ---
Used features for training: top_hat_box=30 top_hat_box=20 top_hat_box=15 laplace_box_of_gaussian_blur=15 difference_of_gaussian=15 gaussian_blur=10 top_hat_box=10 small_hessian_eigenvalue_of_gaussian_blur=10 maximum_box=3 difference_of_gaussian=20 maximum_box=2 gaussian_blur=5 difference_of_gaussian=10 median_box=5 mean_box=10 gaussian_blur=15 sobel_of_gaussian_blur=10 laplace_box_of_gaussian_blur=10 sobel_of_median_box=5 mean_box=15 maximum_box=5 median_box=3 laplace_box_of_gaussian_blur=20 variance_box=3 maximum_box=30 maximum_box=1 variance_box=30 variance_box=2 top_hat_box=2 maximum_box=10 maximum_box=20 maximum_box=15 gaussian_blur=3 variance_box=15 variance_box=10 top_hat_box=3 difference_of_gaussian=2 variance_box=20 laplace_box_of_gaussian_blur=2 sobel_of_gaussian_blur=5 difference_of_gaussian=30 laplace_box_of_gaussian_blur=1 small_hessian_eigenvalue_of_gaussian_blur=15 gaussian_blur=20
Ground truth dimensions: 2
Maxi